# DAA4: Write a program to solve a 0-1 Knapsack problem using dynamic
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;programming or branch and bound strategy.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(Both methods are implemented below.)  

## Theory

1. 0/1 KNAPSACK  
&nbsp;&nbsp;&nbsp;n items with weight w\_i and profit p\_i, knapsack capacity W.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;maximise SUM p\_i x\_i   subject to SUM w\_i x\_i &lt;= W,  x\_i in {0,1}  
&nbsp;&nbsp;&nbsp;Items cannot be broken, so the greedy ratio method may fail.  
&nbsp;&nbsp;&nbsp;Brute force checks all 2^n subsets -&gt; O(2^n). 0/1 knapsack is NP-hard  
&nbsp;&nbsp;&nbsp;(decision version NP-complete); DP gives a PSEUDO-POLYNOMIAL O(nW) solution.  

### 2. DYNAMIC PROGRAMMING (Unit III)
&nbsp;&nbsp;&nbsp;Principle: solve each sub-problem once, store its answer in a table and  
&nbsp;&nbsp;&nbsp;reuse it. Applicable when the problem has  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a) OPTIMAL SUBSTRUCTURE  (Principle of Optimality - Bellman: whatever the  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;initial state and decision, the remaining decisions must be optimal  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;with respect to the state resulting from the first decision)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;b) OVERLAPPING SUB-PROBLEMS.  
&nbsp;&nbsp;&nbsp;Approaches: top-down (memoization) or bottom-up (tabulation).  
&nbsp;&nbsp;&nbsp;Control abstraction:  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Algorithm DP(problem)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;characterise structure of optimal solution  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;define value of optimal solution recursively  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;compute values bottom-up in a table  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;construct optimal solution from computed information (traceback)  

### 3. DP RECURRENCE FOR 0/1 KNAPSACK
&nbsp;&nbsp;&nbsp;K[i][w] = max profit using first i items with capacity w  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;K[0][w] = 0, K[i][0] = 0  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;K[i][w] = K[i-1][w]                               if w\_i &gt; w  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= max( K[i-1][w], p\_i + K[i-1][w - w\_i] )  otherwise  
&nbsp;&nbsp;&nbsp;Answer = K[n][W]. Traceback: if K[i][w] != K[i-1][w], item i is taken.  
&nbsp;&nbsp;&nbsp;Time O(nW), Space O(nW)  (can be reduced to O(W) with a 1-D array).  

### 4. BRANCH AND BOUND (Unit IV)
&nbsp;&nbsp;&nbsp;Explores a STATE-SPACE TREE (each level = decision include/exclude item i)  
&nbsp;&nbsp;&nbsp;but, unlike backtracking, uses a BOUND function to prune nodes that cannot  
&nbsp;&nbsp;&nbsp;lead to a better solution than the best found so far.  
&nbsp;&nbsp;&nbsp;Terms: live node (generated, children not yet all generated), E-node  
&nbsp;&nbsp;&nbsp;(node being expanded), dead node (fully expanded or pruned).  
&nbsp;&nbsp;&nbsp;Strategies to choose next E-node:  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- FIFO B&amp;B  : queue (BFS)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- LIFO B&amp;B  : stack (DFS)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- LC  (Least Cost) B&amp;B : priority queue on cost/bound (best-first)  
&nbsp;&nbsp;&nbsp;For maximisation, the UPPER BOUND of a node = current profit + profit of  
&nbsp;&nbsp;&nbsp;remaining items filled greedily by ratio (fractional knapsack relaxation).  
&nbsp;&nbsp;&nbsp;If bound &lt;= best profit found -&gt; prune.  
&nbsp;&nbsp;&nbsp;Worst case still O(2^n) but pruning makes it much faster in practice.  

### 5. EXAMPLE  W = 50, p = (60, 100, 120), w = (10, 20, 30)
&nbsp;&nbsp;&nbsp;Greedy-by-ratio gives 160 (items 1,2) but optimal is 220 (items 2,3).  

### 6. BACKTRACKING vs BRANCH &amp; BOUND

| Backtracking                      | Branch and Bound                 |
|-----------------------------------|----------------------------------|
| DFS of state-space tree           | BFS / LC search                  |
| used for decision problems        | used for optimisation problems   |
| prunes on constraint violation    | prunes on bound value too        |

### Conclusion
DP solves 0/1 knapsack exactly in O(nW); LC Branch &amp; Bound with a
fractional upper bound also finds the optimum while pruning the search tree.

In [1]:
import heapq

### Method 1: Dynamic Programming (bottom-up tabulation)

In [2]:
def knapsack_dp(profits, weights, W):
    n = len(profits)
    K = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(W + 1):
            if weights[i - 1] > w:
                K[i][w] = K[i - 1][w]
            else:
                K[i][w] = max(K[i - 1][w],
                              profits[i - 1] + K[i - 1][w - weights[i - 1]])

    # traceback to find selected items
    selected, w = [], W
    for i in range(n, 0, -1):
        if K[i][w] != K[i - 1][w]:
            selected.append(i)
            w -= weights[i - 1]
    return K[n][W], sorted(selected), K

### Method 2: Least-Cost Branch and Bound

In [3]:
def knapsack_bnb(profits, weights, W):
    n = len(profits)
    # sort by ratio so that the fractional bound is easy to compute
    order = sorted(range(n), key=lambda i: profits[i] / weights[i],
                   reverse=True)
    p = [profits[i] for i in order]
    w = [weights[i] for i in order]

    def bound(level, cur_w, cur_p):
        """upper bound: fill remaining capacity greedily (fractional)"""
        if cur_w >= W:
            return 0
        b, tw = cur_p, cur_w
        j = level
        while j < n and tw + w[j] <= W:
            tw += w[j]
            b += p[j]
            j += 1
        if j < n:
            b += (W - tw) * p[j] / w[j]
        return b

    best_profit, best_set = 0, []
    nodes_explored = 0
    # max-heap on bound: (-bound, level, weight, profit, taken_items)
    pq = [(-bound(0, 0, 0), 0, 0, 0, [])]
    while pq:
        neg_b, level, cw, cp, taken = heapq.heappop(pq)
        nodes_explored += 1
        if -neg_b <= best_profit or level == n:
            continue                                  # prune
        # child 1: include item `level`
        nw, np_ = cw + w[level], cp + p[level]
        if nw <= W:
            if np_ > best_profit:
                best_profit, best_set = np_, taken + [order[level] + 1]
            b = bound(level + 1, nw, np_)
            if b > best_profit:
                heapq.heappush(pq, (-b, level + 1, nw, np_,
                                    taken + [order[level] + 1]))
        # child 2: exclude item `level`
        b = bound(level + 1, cw, cp)
        if b > best_profit:
            heapq.heappush(pq, (-b, level + 1, cw, cp, taken))
    return best_profit, sorted(best_set), nodes_explored

### Run the program
Enter your own values when prompted, or just press Enter to use the sample data.

In [4]:
def main():
    try:
        n = int(input("Enter number of items (Enter for sample data): "))
        profits = list(map(int, input("Enter profits: ").split()))
        weights = list(map(int, input("Enter weights: ").split()))
        W = int(input("Enter capacity: "))
        assert len(profits) == len(weights) == n
    except Exception:  # bad input, no input, or notebook without stdin
        profits, weights, W = [60, 100, 120], [10, 20, 30], 50
        print(f"Using sample data: profits={profits} weights={weights} W={W}")

    best, items, K = knapsack_dp(profits, weights, W)
    print("\n--- Dynamic Programming ---")
    if W <= 20:
        print("DP table K[i][w]:")
        for i, row in enumerate(K):
            print(f"i={i}:", row)
    print("Maximum profit :", best)
    print("Items selected :", items)
    print("Time O(nW), Space O(nW)")

    best_b, items_b, explored = knapsack_bnb(profits, weights, W)
    print("\n--- LC Branch and Bound ---")
    print("Maximum profit :", best_b)
    print("Items selected :", items_b)
    print("Nodes explored :", explored, f"(brute force would be {2**len(profits)} subsets)")


if __name__ == "__main__":
    main()

Using sample data: profits=[60, 100, 120] weights=[10, 20, 30] W=50

--- Dynamic Programming ---
Maximum profit : 220
Items selected : [2, 3]
Time O(nW), Space O(nW)

--- LC Branch and Bound ---
Maximum profit : 220
Items selected : [2, 3]
Nodes explored : 6 (brute force would be 8 subsets)
